# Lab 06 — 04 Gold Aggregations

**Dataset:** Synthea Healthcare  
**Layer:** Gold business aggregates

## Purpose

Create business-facing aggregate tables for the AI/BI dashboard and Genie:

- `agg_daily_encounters`
- `agg_organization_performance`
- `agg_payer_performance`
- `agg_condition_summary`

### Design choices

- Aggregations are built only from validated Gold facts and dimensions.
- Shared object names come from `src/config.py`.
- Aggregation grains are explicit and validated.
- Every aggregate reconciles back to its source fact table.
- No hard-coded user/workspace paths.

## 1. Runtime context

Common runtime values come from the Job and `src/runtime_config.py`; no widget definitions are stored in this notebook.

In [ ]:
import sys
from pathlib import Path

lab_root = Path.cwd().parent
if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.runtime_config import load_runtime_context

ctx = load_runtime_context(
    dbutils,
    include_validation=True,
)

catalog = ctx.catalog
schema = ctx.schema
volume_name = ctx.volume_name
config = ctx.config
run_validation = ctx.run_validation

print(f"Catalog        : {catalog}")
print(f"Schema         : {schema}")
print(f"Volume         : {volume_name}")
print(f"Run validation : {run_validation}")

## 2. Shared configuration

In [ ]:
from pyspark.sql import functions as F

print("Aggregate targets:")
print(f"  {config.agg_daily_encounters}")
print(f"  {config.agg_organization_performance}")
print(f"  {config.agg_payer_performance}")
print(f"  {config.agg_condition_summary}")

## 3. Validate required Gold inputs

In [ ]:
REQUIRED_TABLES = {
    "fact_encounters": config.fact_encounters,
    "fact_conditions": config.fact_conditions,
    "dim_date": config.dim_date,
    "dim_organization": config.dim_organization,
    "dim_payer": config.dim_payer,
    "dim_condition": config.dim_condition,
}

table_validation_rows = []
missing_tables = []

for logical_name, table_name in REQUIRED_TABLES.items():
    exists = spark.catalog.tableExists(table_name)

    table_validation_rows.append(
        (
            logical_name,
            table_name,
            "PASS" if exists else "FAIL",
        )
    )

    if not exists:
        missing_tables.append(table_name)

table_validation_df = spark.createDataFrame(
    table_validation_rows,
    ["object", "table_name", "status"],
)

display(table_validation_df)

if missing_tables:
    raise RuntimeError(
        "Missing required Gold tables: "
        + ", ".join(missing_tables)
    )

print("Required Gold inputs are available.")

## 4. Load Gold facts and dimensions

In [ ]:
fact_encounters = spark.table(config.fact_encounters)
fact_conditions = spark.table(config.fact_conditions)

dim_date = spark.table(config.dim_date)
dim_organization = spark.table(config.dim_organization)
dim_payer = spark.table(config.dim_payer)
dim_condition = spark.table(config.dim_condition)

print(f"fact_encounters rows : {fact_encounters.count():,}")
print(f"fact_conditions rows : {fact_conditions.count():,}")

## 5. Build `agg_daily_encounters`

Grain:

> **one row per calendar date**

This is the primary time-series dataset for the dashboard.

In [ ]:
agg_daily_encounters_df = (
    fact_encounters
    .groupBy(
        "date_key",
        "encounter_date",
    )
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.countDistinct("organization_key").alias("organizations_active"),
        F.countDistinct("provider_key").alias("providers_active"),

        F.round(
            F.avg("duration_minutes"),
            2,
        ).alias("avg_duration_minutes"),

        F.round(
            F.sum("base_encounter_cost"),
            2,
        ).alias("base_encounter_cost"),

        F.round(
            F.sum("total_claim_cost"),
            2,
        ).alias("total_claim_cost"),

        F.round(
            F.sum("payer_coverage"),
            2,
        ).alias("payer_coverage"),

        F.round(
            F.sum("patient_responsibility"),
            2,
        ).alias("patient_responsibility"),

        F.sum(
            F.when(
                F.lower("encounter_class") == "emergency",
                1,
            ).otherwise(0)
        ).alias("emergency_encounters"),
    )
    .withColumn(
        "emergency_encounter_pct",
        F.round(
            F.col("emergency_encounters")
            / F.col("encounter_count")
            * F.lit(100.0),
            2,
        ),
    )
)

display(
    agg_daily_encounters_df
    .orderBy("encounter_date")
    .limit(20)
)

In [ ]:
(
    agg_daily_encounters_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.agg_daily_encounters)
)

print(f"Created: {config.agg_daily_encounters}")

## 6. Build `agg_organization_performance`

Grain:

> **one row per healthcare organization**

In [ ]:
agg_organization_performance_df = (
    fact_encounters.alias("f")
    .groupBy("organization_key")
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.countDistinct("provider_key").alias("unique_providers"),

        F.round(
            F.avg("duration_minutes"),
            2,
        ).alias("avg_duration_minutes"),

        F.round(
            F.sum("total_claim_cost"),
            2,
        ).alias("total_claim_cost"),

        F.round(
            F.avg("total_claim_cost"),
            2,
        ).alias("avg_claim_cost"),

        F.round(
            F.sum("payer_coverage"),
            2,
        ).alias("payer_coverage"),

        F.round(
            F.sum("patient_responsibility"),
            2,
        ).alias("patient_responsibility"),

        F.min("encounter_date").alias("first_encounter_date"),
        F.max("encounter_date").alias("last_encounter_date"),
    )
    .join(
        dim_organization.select(
            "organization_key",
            "organization_id",
            "organization_name",
            "city",
            "state",
        ),
        "organization_key",
        "left",
    )
    .select(
        "organization_key",
        "organization_id",
        "organization_name",
        "city",
        "state",
        "encounter_count",
        "unique_patients",
        "unique_providers",
        "avg_duration_minutes",
        "total_claim_cost",
        "avg_claim_cost",
        "payer_coverage",
        "patient_responsibility",
        "first_encounter_date",
        "last_encounter_date",
    )
)

display(
    agg_organization_performance_df
    .orderBy(F.desc("encounter_count"))
    .limit(20)
)

In [ ]:
(
    agg_organization_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.agg_organization_performance)
)

print(f"Created: {config.agg_organization_performance}")

## 7. Build `agg_payer_performance`

Grain:

> **one row per payer**

In [ ]:
agg_payer_performance_df = (
    fact_encounters
    .groupBy("payer_key")
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("patient_key").alias("unique_patients"),

        F.round(
            F.sum("total_claim_cost"),
            2,
        ).alias("total_claim_cost"),

        F.round(
            F.sum("payer_coverage"),
            2,
        ).alias("payer_coverage"),

        F.round(
            F.sum("patient_responsibility"),
            2,
        ).alias("patient_responsibility"),

        F.round(
            F.avg("total_claim_cost"),
            2,
        ).alias("avg_claim_cost"),

        F.round(
            F.avg("payer_coverage"),
            2,
        ).alias("avg_payer_coverage"),
    )
    .withColumn(
        "coverage_pct",
        F.when(
            F.col("total_claim_cost") > 0,
            F.round(
                F.col("payer_coverage")
                / F.col("total_claim_cost")
                * F.lit(100.0),
                2,
            ),
        ),
    )
    .join(
        dim_payer.select(
            "payer_key",
            "payer_id",
            "payer_name",
            "state_headquartered",
        ),
        "payer_key",
        "left",
    )
    .select(
        "payer_key",
        "payer_id",
        "payer_name",
        "state_headquartered",
        "encounter_count",
        "unique_patients",
        "total_claim_cost",
        "payer_coverage",
        "patient_responsibility",
        "avg_claim_cost",
        "avg_payer_coverage",
        "coverage_pct",
    )
)

display(
    agg_payer_performance_df
    .orderBy(F.desc("total_claim_cost"))
)

In [ ]:
(
    agg_payer_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.agg_payer_performance)
)

print(f"Created: {config.agg_payer_performance}")

## 8. Build `agg_condition_summary`

Grain:

> **one row per condition**

In [ ]:
agg_condition_summary_df = (
    fact_conditions
    .groupBy("condition_key")
    .agg(
        F.count("*").alias("condition_event_count"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.countDistinct("encounter_key").alias("linked_encounters"),

        F.sum(
            F.when(
                F.col("is_active_condition"),
                1,
            ).otherwise(0)
        ).alias("active_condition_events"),

        F.round(
            F.avg("condition_duration_days"),
            2,
        ).alias("avg_condition_duration_days"),

        F.min("condition_start_date").alias("first_recorded_date"),
        F.max("condition_start_date").alias("last_recorded_date"),
    )
    .withColumn(
        "active_condition_pct",
        F.round(
            F.col("active_condition_events")
            / F.col("condition_event_count")
            * F.lit(100.0),
            2,
        ),
    )
    .join(
        dim_condition.select(
            "condition_key",
            "condition_code",
            "condition_description",
        ),
        "condition_key",
        "left",
    )
    .select(
        "condition_key",
        "condition_code",
        "condition_description",
        "condition_event_count",
        "unique_patients",
        "linked_encounters",
        "active_condition_events",
        "active_condition_pct",
        "avg_condition_duration_days",
        "first_recorded_date",
        "last_recorded_date",
    )
)

display(
    agg_condition_summary_df
    .orderBy(F.desc("condition_event_count"))
    .limit(25)
)

In [ ]:
(
    agg_condition_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.agg_condition_summary)
)

print(f"Created: {config.agg_condition_summary}")

## 9. Validate aggregate grains

Each aggregate must contain one unique row per declared grain.

In [ ]:
GRAIN_CHECKS = {
    "agg_daily_encounters": (
        config.agg_daily_encounters,
        ["date_key"],
    ),
    "agg_organization_performance": (
        config.agg_organization_performance,
        ["organization_key"],
    ),
    "agg_payer_performance": (
        config.agg_payer_performance,
        ["payer_key"],
    ),
    "agg_condition_summary": (
        config.agg_condition_summary,
        ["condition_key"],
    ),
}

grain_rows = []
grain_failures = []

for aggregate_name, (
    table_name,
    grain_columns,
) in GRAIN_CHECKS.items():

    df = spark.table(table_name)

    row_count = df.count()

    duplicate_grain_rows = (
        df.groupBy(*grain_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_grain_rows = (
        df.filter(
            F.expr(
                " OR ".join(
                    f"`{column}` IS NULL"
                    for column in grain_columns
                )
            )
        )
        .count()
    )

    status = (
        "PASS"
        if duplicate_grain_rows == 0
        and null_grain_rows == 0
        else "FAIL"
    )

    grain_rows.append(
        (
            aggregate_name,
            row_count,
            duplicate_grain_rows,
            null_grain_rows,
            status,
        )
    )

    if status == "FAIL":
        grain_failures.append(aggregate_name)

grain_validation_df = spark.createDataFrame(
    grain_rows,
    [
        "aggregate",
        "row_count",
        "duplicate_grain_rows",
        "null_grain_rows",
        "status",
    ],
)

display(
    grain_validation_df.orderBy("aggregate")
)

if run_validation and grain_failures:
    raise ValueError(
        "Aggregate grain validation failed: "
        + ", ".join(grain_failures)
    )

## 10. Reconcile encounter aggregates to `fact_encounters`

Encounter counts across daily, organization and payer aggregates must each
reconcile to the source fact row count.

In [ ]:
fact_encounter_count = fact_encounters.count()

encounter_reconciliation = {
    "agg_daily_encounters": (
        spark.table(config.agg_daily_encounters)
        .agg(F.sum("encounter_count").alias("value"))
        .first()["value"]
    ),
    "agg_organization_performance": (
        spark.table(config.agg_organization_performance)
        .agg(F.sum("encounter_count").alias("value"))
        .first()["value"]
    ),
    "agg_payer_performance": (
        spark.table(config.agg_payer_performance)
        .agg(F.sum("encounter_count").alias("value"))
        .first()["value"]
    ),
}

encounter_reconciliation_rows = []
encounter_reconciliation_failures = []

for aggregate_name, aggregate_count in encounter_reconciliation.items():
    aggregate_count = int(aggregate_count or 0)

    status = (
        "PASS"
        if aggregate_count == fact_encounter_count
        else "FAIL"
    )

    encounter_reconciliation_rows.append(
        (
            aggregate_name,
            fact_encounter_count,
            aggregate_count,
            status,
        )
    )

    if status == "FAIL":
        encounter_reconciliation_failures.append(
            aggregate_name
        )

encounter_reconciliation_df = spark.createDataFrame(
    encounter_reconciliation_rows,
    [
        "aggregate",
        "fact_encounter_rows",
        "aggregate_encounter_rows",
        "status",
    ],
)

display(encounter_reconciliation_df)

if run_validation and encounter_reconciliation_failures:
    raise ValueError(
        "Encounter aggregation reconciliation failed: "
        + ", ".join(
            encounter_reconciliation_failures
        )
    )

## 11. Reconcile condition aggregate to `fact_conditions`

In [ ]:
fact_condition_count = fact_conditions.count()

condition_aggregate_count = int(
    (
        spark.table(config.agg_condition_summary)
        .agg(
            F.sum("condition_event_count").alias("value")
        )
        .first()["value"]
    )
    or 0
)

condition_reconciliation_status = (
    "PASS"
    if condition_aggregate_count == fact_condition_count
    else "FAIL"
)

condition_reconciliation_df = spark.createDataFrame(
    [(
        fact_condition_count,
        condition_aggregate_count,
        condition_reconciliation_status,
    )],
    [
        "fact_condition_rows",
        "aggregate_condition_rows",
        "status",
    ],
)

display(condition_reconciliation_df)

if (
    run_validation
    and condition_reconciliation_status == "FAIL"
):
    raise ValueError(
        "Condition aggregation reconciliation failed."
    )

## 12. Financial measure reconciliation

Financial totals from business aggregates must equal `fact_encounters`.

In [ ]:
fact_financials = (
    fact_encounters
    .agg(
        F.sum("total_claim_cost").alias("total_claim_cost"),
        F.sum("payer_coverage").alias("payer_coverage"),
        F.sum("patient_responsibility")
            .alias("patient_responsibility"),
    )
    .first()
)

organization_financials = (
    spark.table(config.agg_organization_performance)
    .agg(
        F.sum("total_claim_cost").alias("total_claim_cost"),
        F.sum("payer_coverage").alias("payer_coverage"),
        F.sum("patient_responsibility")
            .alias("patient_responsibility"),
    )
    .first()
)

financial_rows = []
financial_failures = []

for measure in [
    "total_claim_cost",
    "payer_coverage",
    "patient_responsibility",
]:
    fact_value = fact_financials[measure]
    aggregate_value = organization_financials[measure]

    status = (
        "PASS"
        if fact_value == aggregate_value
        else "FAIL"
    )

    financial_rows.append(
        (
            measure,
            str(fact_value),
            str(aggregate_value),
            status,
        )
    )

    if status == "FAIL":
        financial_failures.append(measure)

financial_validation_df = spark.createDataFrame(
    financial_rows,
    [
        "measure",
        "fact_total",
        "organization_aggregate_total",
        "status",
    ],
)

display(financial_validation_df)

if run_validation and financial_failures:
    raise ValueError(
        "Financial aggregation reconciliation failed: "
        + ", ".join(financial_failures)
    )

## 13. Gold aggregate inventory

In [ ]:
AGGREGATE_TABLES = [
    config.agg_daily_encounters,
    config.agg_organization_performance,
    config.agg_payer_performance,
    config.agg_condition_summary,
]

inventory_rows = []

for table_name in AGGREGATE_TABLES:
    df = spark.table(table_name)

    inventory_rows.append(
        (
            table_name.split(".")[-1],
            table_name,
            df.count(),
            len(df.columns),
        )
    )

aggregate_inventory_df = spark.createDataFrame(
    inventory_rows,
    [
        "aggregate",
        "table_name",
        "row_count",
        "column_count",
    ],
)

display(
    aggregate_inventory_df.orderBy("aggregate")
)

## 14. Dashboard preview — organization performance

In [ ]:
display(
    spark.table(config.agg_organization_performance)
    .select(
        "organization_name",
        "state",
        "encounter_count",
        "unique_patients",
        "total_claim_cost",
        "avg_claim_cost",
        "avg_duration_minutes",
    )
    .orderBy(F.desc("encounter_count"))
    .limit(20)
)

## 15. Dashboard preview — payer performance

In [ ]:
display(
    spark.table(config.agg_payer_performance)
    .select(
        "payer_name",
        "encounter_count",
        "unique_patients",
        "total_claim_cost",
        "payer_coverage",
        "patient_responsibility",
        "coverage_pct",
    )
    .orderBy(F.desc("total_claim_cost"))
)

## 16. Final validation

In [ ]:
final_checks = [
    ("aggregate_grains", len(grain_failures) == 0),
    (
        "encounter_reconciliation",
        len(encounter_reconciliation_failures) == 0,
    ),
    (
        "condition_reconciliation",
        condition_reconciliation_status == "PASS",
    ),
    (
        "financial_reconciliation",
        len(financial_failures) == 0,
    ),
]

final_validation_df = spark.createDataFrame(
    [
        (
            check_name,
            "PASS" if passed else "FAIL",
        )
        for check_name, passed in final_checks
    ],
    ["check_name", "status"],
)

display(final_validation_df)

failed_checks = [
    check_name
    for check_name, passed in final_checks
    if not passed
]

if run_validation and failed_checks:
    raise RuntimeError(
        "Gold aggregation validation failed: "
        + ", ".join(failed_checks)
    )

## 17. Completion

Created:

```text
agg_daily_encounters
agg_organization_performance
agg_payer_performance
agg_condition_summary
```

These tables provide dashboard-ready business metrics while retaining
reconciliation to the underlying Gold facts.

**Next:** AI/BI dashboard, followed by governance.

In [ ]:
print("LAB 06 — GOLD AGGREGATIONS COMPLETE")
print("")
print(f"Created: {config.agg_daily_encounters}")
print(f"Created: {config.agg_organization_performance}")
print(f"Created: {config.agg_payer_performance}")
print(f"Created: {config.agg_condition_summary}")
print("")
print("Next: AI/BI dashboard")